## Accelerate Inference: Neural Network Pruning

In [1]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

dataset.tar.gz	train_images.pkl  val_images.pkl
sample_data	train_labels.pkl  val_labels.pkl
train_images.pkl
train_labels.pkl
val_images.pkl
val_labels.pkl


In [4]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

In [5]:
train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [7]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [8]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [9]:
model = model.to(device)
summary(model, input_size=(3, 25, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 25, 25]             896
              ReLU-2           [-1, 32, 25, 25]               0
            Conv2d-3           [-1, 32, 23, 23]           9,248
              ReLU-4           [-1, 32, 23, 23]               0
         MaxPool2d-5           [-1, 32, 11, 11]               0
           Dropout-6           [-1, 32, 11, 11]               0
            Conv2d-7           [-1, 64, 11, 11]          18,496
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9             [-1, 64, 9, 9]          36,928
             ReLU-10             [-1, 64, 9, 9]               0
        MaxPool2d-11             [-1, 64, 4, 4]               0
          Dropout-12             [-1, 64, 4, 4]               0
          Flatten-13                 [-1, 1024]               0
           Linear-14                  [

In [10]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [11]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

In [12]:
# Main training loop
num_epochs = 50
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Training
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

Epoch 1/50


Epoch [1/50], Train Loss: 1.5006, Train Acc: 32.17%, Val Loss: 1.3682, Val Acc: 41.70%
Epoch 2/50


Epoch [2/50], Train Loss: 1.3615, Train Acc: 41.58%, Val Loss: 1.3066, Val Acc: 44.91%
Epoch 3/50


Epoch [3/50], Train Loss: 1.3188, Train Acc: 43.79%, Val Loss: 1.2624, Val Acc: 46.22%
Epoch 4/50


Epoch [4/50], Train Loss: 1.2836, Train Acc: 45.94%, Val Loss: 1.2410, Val Acc: 47.09%
Epoch 5/50


...

Epoch [46/50], Train Loss: 0.7422, Train Acc: 71.59%, Val Loss: 0.8347, Val Acc: 68.04%
Epoch 47/50


Epoch [47/50], Train Loss: 0.7250, Train Acc: 72.32%, Val Loss: 0.8387, Val Acc: 67.13%
Epoch 48/50


Epoch [48/50], Train Loss: 0.7220, Train Acc: 72.58%, Val Loss: 0.8206, Val Acc: 67.60%
Epoch 49/50


Epoch [49/50], Train Loss: 0.7145, Train Acc: 72.70%, Val Loss: 0.8025, Val Acc: 68.79%
Epoch 50/50


Epoch [50/50], Train Loss: 0.7010, Train Acc: 73.61%, Val Loss: 0.8116, Val Acc: 68.51%


In [13]:
torch.save(model.state_dict(), 'my_model_weights_1.pt', _use_new_zipfile_serialization=False)

### GMP

Gradual Magnitude Pruning (GMP) progressively increases sparsity during training using a polynomial schedule.

**Reference:** Zhu & Gupta (2017) - "To prune, or not to prune: exploring the efficacy of pruning for model compression"

**Key differences from one-shot pruning:**
- Gradually increases sparsity from 0% to target over multiple epochs
- Uses polynomial schedule: $s_t = s_f + (s_i - s_f)(1 - \frac{t-t_0}{n\Delta t})^3$
- Model adapts to increasing sparsity during training

In [14]:
def compute_sparsity(model):
    """Calculate overall sparsity of the model"""
    total_zeros = sum(torch.sum(p == 0).item() for p in model.parameters())
    total_params = sum(p.numel() for p in model.parameters())
    return total_zeros / total_params

def polynomial_schedule(initial_sparsity, final_sparsity, t, t_start, t_end, exponent=3):
    """Compute sparsity at time t using polynomial schedule"""
    if t <= t_start:
        return initial_sparsity
    elif t >= t_end:
        return final_sparsity
    else:
        progress = (t - t_start) / (t_end - t_start)
        return final_sparsity + (initial_sparsity - final_sparsity) * (1 - progress) ** exponent

def apply_magnitude_pruning(model, target_sparsity):
    """Apply global magnitude-based pruning and return mask"""
    mask = {}

    with torch.no_grad():
        # Collect all weights (not biases)
        all_weights = []
        for name, param in model.named_parameters():
            if "weight" in name:
                all_weights.append(param.abs().flatten())

        # Concatenate and calculate global threshold
        all_weights = torch.cat(all_weights)
        threshold = torch.quantile(all_weights, target_sparsity)

        # Apply pruning to each layer and create mask
        for name, param in model.named_parameters():
            if "weight" in name:
                m = (param.abs() > threshold).float().to(device)
                param.mul_(m)
                mask[name] = m

    return mask, threshold

def train_one_epoch_with_pruning(model, train_loader, optimizer, criterion, device, mask=None):
    """Training function with mask enforcement"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # CRITICAL: Re-apply mask after optimizer step
        if mask is not None:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in mask:
                        param.mul_(mask[name])

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

# Load the trained model
model.load_state_dict(torch.load('my_model_weights_1.pt'))
model = model.to(device)

# GMP hyperparameters
num_epochs = 30
initial_sparsity = 0.0
final_sparsity = 0.70
prune_start_epoch = 5
prune_end_epoch = 25
prune_frequency = 2

# Reset optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

print("="*70)
print("Gradual Magnitude Pruning Configuration:")
print(f"  Total epochs: {num_epochs}")
print(f"  Pruning window: epochs {prune_start_epoch} to {prune_end_epoch}")
print(f"  Target sparsity: {initial_sparsity:.2%} -> {final_sparsity:.2%}")
print(f"  Pruning frequency: every {prune_frequency} epochs")
print("="*70)

best_val_accuracy = 0.0
mask = None  # Initialize mask

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # Calculate current target sparsity
    current_sparsity_target = polynomial_schedule(
        initial_sparsity, final_sparsity,
        epoch, prune_start_epoch, prune_end_epoch
    )

    # Apply pruning at specified frequency
    if epoch >= prune_start_epoch and epoch % prune_frequency == 0:
        print(f"  Applying pruning to target sparsity: {current_sparsity_target:.4f}")
        mask, threshold = apply_magnitude_pruning(model, current_sparsity_target)
        actual_sparsity = compute_sparsity(model)
        print(f"  Threshold: {threshold:.6f}")
        print(f"  Actual sparsity after pruning: {actual_sparsity:.4f}")

    # Training with mask enforcement
    train_loss, train_accuracy = train_one_epoch_with_pruning(
        model, train_loader, optimizer, criterion, device, mask
    )

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Calculate current sparsity and score
    current_sparsity = compute_sparsity(model)
    score = (val_accuracy/100 + current_sparsity) / 2 if val_accuracy > 60 else 0

    # Print epoch results
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
    print(f'  Sparsity: {current_sparsity:.4f}, Score: {score:.4f}')

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        print(f"  ✓ New best validation accuracy: {val_accuracy:.2f}%")

final_sparsity = compute_sparsity(model)
final_val_loss, final_val_accuracy = validate(model, val_loader, criterion, device)
final_score = (final_val_accuracy/100 + final_sparsity) / 2 if final_val_accuracy > 60 else 0

print(f"\n{'='*70}")
print(f"Final Results:")
print(f"  Validation Accuracy: {final_val_accuracy:.2f}%")
print(f"  Sparsity: {final_sparsity:.4f}")
print(f"  Score: {final_score:.4f}")
print(f"{'='*70}\n")

Gradual Magnitude Pruning Configuration:
  Total epochs: 30
  Pruning window: epochs 5 to 25
  Target sparsity: 0.00% -> 70.00%
  Pruning frequency: every 2 epochs

Epoch 1/30


  Train Loss: 0.6982, Train Acc: 73.49%
  Val Loss: 0.8038, Val Acc: 68.55%
  Sparsity: 0.0000, Score: 0.3428
  ✓ New best validation accuracy: 68.55%

Epoch 2/30


  Train Loss: 0.6873, Train Acc: 73.95%
  Val Loss: 0.8117, Val Acc: 68.24%
  Sparsity: 0.0000, Score: 0.3412

Epoch 3/30


  Train Loss: 0.6797, Train Acc: 74.37%
  Val Loss: 0.8056, Val Acc: 69.11%
  Sparsity: 0.0000, Score: 0.3455
  ✓ New best validation accuracy: 69.11%

Epoch 4/30


  Train Loss: 0.6766, Train Acc: 74.10%
  Val Loss: 0.7971, Val Acc: 68.91%
  Sparsity: 0.0000, Score: 0.3446

Epoch 5/30


...

  Train Loss: 0.5449, Train Acc: 79.36%
  Val Loss: 0.7654, Val Acc: 71.64%
  Sparsity: 0.6992, Score: 0.7078
  ✓ New best validation accuracy: 71.64%

Epoch 28/30


  Train Loss: 0.5354, Train Acc: 79.76%
  Val Loss: 0.7666, Val Acc: 71.60%
  Sparsity: 0.6992, Score: 0.7076

Epoch 29/30
  Applying pruning to target sparsity: 0.7000
  Threshold: 0.001256
  Actual sparsity after pruning: 0.6992


  Train Loss: 0.5369, Train Acc: 79.56%
  Val Loss: 0.7605, Val Acc: 71.41%
  Sparsity: 0.6992, Score: 0.7066

Epoch 30/30


  Train Loss: 0.5336, Train Acc: 80.08%
  Val Loss: 0.7606, Val Acc: 71.25%
  Sparsity: 0.6992, Score: 0.7058



Final Results:
  Validation Accuracy: 71.25%
  Sparsity: 0.6992
  Score: 0.7058



In [15]:
# val_loss, val_accuracy = validate(model, val_loader, criterion, device)

In [16]:
# val_accuracy

In [17]:
# torch.save(model.state_dict(), 'my_model_weights_2.pt', _use_new_zipfile_serialization=False)
torch.save(model.state_dict(), 'my_model_weights_2.pt', _use_new_zipfile_serialization=False)
print("Model saved as my_model_weights_2.pt")
print(f"Final Sparsity: {compute_sparsity(model):.4f}")
print(f"Final Accuracy: {final_val_accuracy:.2f}%")
print(f"Final Score: {final_score:.4f}")

Model saved as my_model_weights_2.pt
Final Sparsity: 0.6992
Final Accuracy: 71.25%
Final Score: 0.7058


In [18]:
# Different configurations to try
configurations = [
    {'num_epochs': 30, 'final_sparsity': 0.65, 'prune_start': 5, 'prune_end': 25},
    {'num_epochs': 30, 'final_sparsity': 0.70, 'prune_start': 5, 'prune_end': 25},
    {'num_epochs': 35, 'final_sparsity': 0.70, 'prune_start': 5, 'prune_end': 30},
    {'num_epochs': 30, 'final_sparsity': 0.75, 'prune_start': 5, 'prune_end': 25},
]

best_score = 0
best_config = None
best_state = None

for config in configurations:
    print(f"\n{'='*70}")
    print(f"Testing config: {config}")
    print(f"{'='*70}")

    # Reload original trained model
    model.load_state_dict(torch.load('my_model_weights_1.pt'))
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

    mask = None

    # Run GMP with this configuration
    for epoch in range(config['num_epochs']):
        current_sparsity_target = polynomial_schedule(
            0.0, config['final_sparsity'],
            epoch, config['prune_start'], config['prune_end']
        )

        # Apply pruning and update mask
        if epoch >= config['prune_start'] and epoch % 2 == 0:
            mask, threshold = apply_magnitude_pruning(model, current_sparsity_target)

        # Training with mask enforcement
        train_loss, train_acc = train_one_epoch_with_pruning(
            model, train_loader, optimizer, criterion, device, mask  # Pass mask
        )

        # Periodic validation
        if (epoch + 1) % 10 == 0:
            val_loss, val_acc = validate(model, val_loader, criterion, device)
            sparsity = compute_sparsity(model)
            print(f"  Epoch {epoch+1}: Val Acc: {val_acc:.2f}%, Sparsity: {sparsity:.4f}")

    # Final evaluation
    final_sparsity = compute_sparsity(model)
    _, final_acc = validate(model, val_loader, criterion, device)
    score = (final_acc/100 + final_sparsity) / 2 if final_acc > 60 else 0

    print(f"\nResults:")
    print(f"  Val Acc: {final_acc:.2f}%")
    print(f"  Sparsity: {final_sparsity:.4f}")
    print(f"  Score: {score:.4f}")

    if score > best_score:
        best_score = score
        best_config = config
        best_state = model.state_dict().copy()  # Save best model state
        print(f"  ✓ New best score!")

print(f"\n{'='*70}")
print(f"Best Configuration:")
print(f"  Config: {best_config}")
print(f"  Score: {best_score:.4f}")
print(f"{'='*70}")

# Save best model (overwrite my_model_weights_2_best.pt with best config)
if best_state is not None:
    torch.save(best_state, 'my_model_weights_2_best.pt', _use_new_zipfile_serialization=False)
    print("\nBest model saved as my_model_weights_2_best.pt")


Testing config: {'num_epochs': 30, 'final_sparsity': 0.65, 'prune_start': 5, 'prune_end': 25}


  Epoch 10: Val Acc: 69.62%, Sparsity: 0.2505


  Epoch 20: Val Acc: 70.89%, Sparsity: 0.6214


  Epoch 30: Val Acc: 71.21%, Sparsity: 0.6492



Results:
  Val Acc: 71.21%
  Sparsity: 0.6492
  Score: 0.6807
  ✓ New best score!

Testing config: {'num_epochs': 30, 'final_sparsity': 0.7, 'prune_start': 5, 'prune_end': 25}


...


Results:
  Val Acc: 70.81%
  Sparsity: 0.6992
  Score: 0.7036

Testing config: {'num_epochs': 30, 'final_sparsity': 0.75, 'prune_start': 5, 'prune_end': 25}


  Epoch 10: Val Acc: 70.14%, Sparsity: 0.2891


  Epoch 20: Val Acc: 70.93%, Sparsity: 0.7170


  Epoch 30: Val Acc: 71.60%, Sparsity: 0.7491



Results:
  Val Acc: 71.60%
  Sparsity: 0.7491
  Score: 0.7326
  ✓ New best score!

Best Configuration:
  Config: {'num_epochs': 30, 'final_sparsity': 0.75, 'prune_start': 5, 'prune_end': 25}
  Score: 0.7326

Best model saved as my_model_weights_2_best.pt
